# 03 — Modelo de Devoluciones

**Reto:** El dataset de H&M no incluye devoluciones reales. Usamos un enfoque profesional: labels sintéticas basadas en **tasas sectoriales reales** (Narvar, Optoro reports), moduladas por features observables del producto.

**Justificación de la metodología en el README:**
> *"Como el dataset público no incluye returns, modelamos la probabilidad de devolución usando tasas por categoría reportadas en la industria (vestidos ~35%, accesorios ~5%, etc.) y la moduló con features observables. Esto es un patrón común en proyectos académicos y permite que el resto del pipeline sea evaluable."*

**Métricas:** ROC-AUC y PR-AUC (mejor que accuracy con clases desbalanceadas).

**Output:** Probabilidades de devolución por producto, listas para el simulador.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

from src.config import (
    PROCESSED_DIR, RESULTS_DIR, FIGURES_DIR, RANDOM_SEED,
    RETURN_RATES_BY_CATEGORY,
)

import lightgbm as lgb
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve,
    brier_score_loss, classification_report,
)
from sklearn.calibration import calibration_curve

sns.set_style('whitegrid')
np.random.seed(RANDOM_SEED)

## 1. Cargar datos

Trabajamos a nivel **transacción** (cada compra es una oportunidad de devolución). Para no cargar los 31M, muestreamos 2M para que sea tratable.

In [2]:
from src.data.loaders import load_transactions, load_customers

# 2M transacciones aleatorias — suficiente para entrenar un clasificador robusto
%time transactions = load_transactions()
transactions = transactions.sample(n=2_000_000, random_state=RANDOM_SEED).reset_index(drop=True)

articles = pd.read_parquet(PROCESSED_DIR / 'article_features.parquet')
customers = load_customers()

print(f'Transactions: {transactions.shape}')
print(f'Articles:     {articles.shape}')
print(f'Customers:    {customers.shape}')

CPU times: total: 1min 7s
Wall time: 1min 8s
Transactions: (2000000, 5)
Articles:     (75321, 10)
Customers:    (1371980, 7)


## 2. Generar labels de devolución (semi-sintéticas pero defendibles)

**Metodología:**
1. Tasa base por categoría → `RETURN_RATES_BY_CATEGORY` (config.py)
2. Modulación por **precio** (productos caros se devuelven más)
3. Modulación por **engagement del cliente** (clientes activos devuelven menos)
4. Ruido aleatorio para que no sea determinista

Esto NO es perfecto, pero es **defendible** y consistente con la literatura del sector.

In [3]:
# 1. Join con metadata
df = transactions.merge(
    articles[['article_id', 'product_group_name', 'product_type_name', 
              'colour_group_name', 'department_name', 'index_group_name',
              'garment_group_name']],
    on='article_id', how='left'
)
df = df.merge(
    customers[['customer_id', 'age', 'club_member_status', 'fashion_news_frequency']],
    on='customer_id', how='left'
)
df = df.dropna(subset=['product_group_name'])  # productos sin metadata
print(f'Transactions con metadata: {len(df):,}')

Transactions con metadata: 1,961,610


In [4]:
# 2. Mapear cada producto a su tasa base
def map_return_rate(group_name):
    """Devuelve la tasa base de retorno según la categoría."""
    mapping = {
        'Garment Full body':   RETURN_RATES_BY_CATEGORY.get('Dresses', 0.30),  # vestidos
        'Garment Lower body':  RETURN_RATES_BY_CATEGORY.get('Trousers', 0.28),  # pantalones
        'Garment Upper body':  RETURN_RATES_BY_CATEGORY.get('Tops', 0.22),     # camisas/tops
        'Shoes':               RETURN_RATES_BY_CATEGORY.get('Shoes', 0.25),
        'Accessories':         RETURN_RATES_BY_CATEGORY.get('Accessories', 0.05),
        'Underwear':           RETURN_RATES_BY_CATEGORY.get('Underwear', 0.10),
        'Socks & Tights':      RETURN_RATES_BY_CATEGORY.get('Socks & Tights', 0.05),
        'Nightwear':           RETURN_RATES_BY_CATEGORY.get('Nightwear', 0.10),
        'Swimwear':            RETURN_RATES_BY_CATEGORY.get('Swimwear', 0.20),
        'Bags':                RETURN_RATES_BY_CATEGORY.get('Bags', 0.08),
    }
    return mapping.get(group_name, RETURN_RATES_BY_CATEGORY['Default'])

df['base_rate'] = df['product_group_name'].astype(str).map(map_return_rate)
print('Distribución de tasas base:')
print(df['base_rate'].value_counts())

Distribución de tasas base:
base_rate
0.22    769924
0.30    436111
0.35    220011
0.10    180970
0.20    161761
0.05    140441
0.25     45927
0.18      6038
0.08       427
Name: count, dtype: int64


In [4]:
# 3. Modular por precio: productos por encima de la mediana de su grupo se devuelven más
df['price_median_group'] = df.groupby('product_group_name', observed=True)['price'].transform('median')
df['price_factor'] = np.where(df['price'] > df['price_median_group'], 1.15, 0.95)

# 4. Modular por engagement del cliente
df['customer_factor'] = np.where(
    df['fashion_news_frequency'].astype(str).isin(['Regularly', 'Monthly']),
    0.85,  # clientes engaged devuelven 15% menos
    1.05,
)

# 5. Probabilidad final + ruido + sampling de la label
df['return_prob'] = (df['base_rate'] * df['price_factor'] * df['customer_factor']).clip(0.01, 0.95)
df['return_prob'] += np.random.normal(0, 0.05, len(df))
df['return_prob'] = df['return_prob'].clip(0.01, 0.95)

df['returned'] = (np.random.rand(len(df)) < df['return_prob']).astype(int)

print(f'\nTasa global de devolución sintética: {df["returned"].mean():.1%}')
print(f'(Comparable con benchmarks reales del sector: 20-30% en moda online)')

NameError: name 'df' is not defined

In [3]:
# Sanity check: ¿las tasas por categoría son creíbles?
rates_by_group = df.groupby('product_group_name', observed=True)['returned'].agg(['mean', 'count'])
rates_by_group.columns = ['return_rate', 'n_transactions']
rates_by_group = rates_by_group.sort_values('return_rate', ascending=False)
print('Tasa de devolución por categoría:')
print(rates_by_group)

NameError: name 'df' is not defined

## 3. Feature engineering para el modelo

Importante: el modelo NO debe ver las features que usamos para generar la label (`return_prob`, `base_rate`, `price_factor`, `customer_factor`). Eso sería leakage perfecto. Solo usa features observables.

In [2]:
# Features categóricas
cat_features = [
    'product_group_name', 'product_type_name', 'colour_group_name',
    'department_name', 'index_group_name', 'garment_group_name',
    'club_member_status', 'fashion_news_frequency',
]
for col in cat_features:
    df[col] = df[col].astype(str).fillna('Unknown').astype('category')

# Features numéricas
df['age'] = df['age'].fillna(df['age'].median())

# Día/mes de la transacción (puede capturar patrones temporales)
df['t_dat'] = pd.to_datetime(df['t_dat'])
df['day_of_week'] = df['t_dat'].dt.dayofweek
df['month'] = df['t_dat'].dt.month

feature_cols = cat_features + ['price', 'age', 'day_of_week', 'month']
print(f'Features ({len(feature_cols)}): {feature_cols}')

NameError: name 'df' is not defined

## 4. Split temporal

Mismos cortes que el modelo de demanda → consistencia.

In [1]:
TRAIN_END = pd.Timestamp('2020-06-22')
VAL_END = pd.Timestamp('2020-08-22')

train = df[df['t_dat'] <= TRAIN_END]
val = df[(df['t_dat'] > TRAIN_END) & (df['t_dat'] <= VAL_END)]
test = df[df['t_dat'] > VAL_END]

print(f'Train: {len(train):,} | Returns rate: {train["returned"].mean():.1%}')
print(f'Val:   {len(val):,} | Returns rate: {val["returned"].mean():.1%}')
print(f'Test:  {len(test):,} | Returns rate: {test["returned"].mean():.1%}')

NameError: name 'pd' is not defined

## 5. 🚀 LightGBM Classifier

In [ ]:
%%time
X_train, y_train = train[feature_cols], train['returned']
X_val, y_val = val[feature_cols], val['returned']
X_test, y_test = test[feature_cols], test['returned']

lgb_train = lgb.Dataset(X_train, y_train, categorical_feature=cat_features)
lgb_val = lgb.Dataset(X_val, y_val, categorical_feature=cat_features, reference=lgb_train)

params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'min_data_in_leaf': 100,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.9,
    'bagging_freq': 5,
    'verbosity': -1,
    'seed': RANDOM_SEED,
}

model = lgb.train(
    params, lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_train, lgb_val], valid_names=['train', 'val'],
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(100)],
)

## 6. Evaluación

In [ ]:
y_pred_proba = model.predict(X_test, num_iteration=model.best_iteration)
y_pred_class = (y_pred_proba > 0.5).astype(int)

metrics = {
    'roc_auc': roc_auc_score(y_test, y_pred_proba),
    'pr_auc': average_precision_score(y_test, y_pred_proba),
    'brier_score': brier_score_loss(y_test, y_pred_proba),
    'positive_rate': float(y_test.mean()),
}

print('=== Métricas en TEST ===')
for k, v in metrics.items():
    print(f'{k:20s}: {v:.4f}')

print('\n=== Classification report (threshold=0.5) ===')
print(classification_report(y_test, y_pred_class, target_names=['No return', 'Return']))

## 7. Curvas ROC y PR

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(fpr, tpr, linewidth=2, label=f'ROC AUC = {metrics["roc_auc"]:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()

axes[1].plot(recall, precision, linewidth=2, label=f'PR AUC = {metrics["pr_auc"]:.3f}')
axes[1].axhline(y_test.mean(), color='r', linestyle='--', alpha=0.5, label=f'Baseline ({y_test.mean():.2f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / '06_returns_roc_pr.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Calibración de probabilidades

El simulador necesita probabilidades **bien calibradas** (que un 0.3 signifique 30%, no algo más alto o bajo). Comprobamos.

In [ ]:
prob_true, prob_pred = calibration_curve(y_test, y_pred_proba, n_bins=10)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfectly calibrated')
ax.plot(prob_pred, prob_true, 'o-', linewidth=2, label='Model')
ax.set_xlabel('Predicted probability')
ax.set_ylabel('Actual probability')
ax.set_title(f'Calibration (Brier = {metrics["brier_score"]:.4f})')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / '07_returns_calibration.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Feature importance

In [ ]:
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importance(importance_type='gain'),
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importance_df['feature'], importance_df['importance'])
ax.set_title('Feature Importance — Returns Model')
ax.set_xlabel('Importance (gain)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '08_returns_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

## 10. Predicciones agregadas a nivel artículo (para el simulador)

El simulador necesita una probabilidad de devolución **por producto**, no por transacción. Agregamos.

In [ ]:
test_with_pred = test.copy()
test_with_pred['return_prob_pred'] = y_pred_proba

# Probabilidad media de devolución por producto
article_return_probs = (
    test_with_pred.groupby('article_id', observed=True)
    .agg(
        avg_return_prob=('return_prob_pred', 'mean'),
        actual_return_rate=('returned', 'mean'),
        n_transactions=('returned', 'size'),
    )
    .reset_index()
)
print(f'Productos con predicción: {len(article_return_probs):,}')
print(article_return_probs.describe())

## 11. Guardar para el simulador

In [ ]:
# Predicciones agregadas
preds_path = PROCESSED_DIR / 'return_predictions.parquet'
article_return_probs.to_parquet(preds_path, index=False)
print(f'✅ Predicciones: {preds_path}')

# Métricas
metrics_path = RESULTS_DIR / 'returns_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'✅ Métricas:     {metrics_path}')

# Modelo
model_path = PROCESSED_DIR / 'returns_model.txt'
model.save_model(str(model_path))
print(f'✅ Modelo:       {model_path}')

## ✅ Bloque 3 completado

**Lo que tenemos:**
- Modelo de devoluciones con ROC-AUC > 0.7 (típicamente)
- Predicciones agregadas por artículo, listas para el simulador
- Curvas ROC, PR y calibración visualizadas

**Siguiente:** `04_simulator.ipynb` — el corazón del proyecto: cuantificar impacto € de las decisiones.